# PriMock57 medical ASR evaluation: full run on a Colab GPU

Runs the whole pipeline (download, prep, transcribe, evaluate) on all 57 consultations
with faster-whisper **large-v3**, under three conditions: doctor channel, patient channel,
and a mixed mono "room mic" signal.

1. *Runtime > Change runtime type > GPU* (T4 works; L4/A100 are faster).
2. Run the cells in order. Outputs go to `WORK` (mount Drive so a disconnect doesn't lose them).
   `transcribe.py` skips files that are already done, so you can just re-run after a disconnect.

Data: PriMock57 (Babylon Health), CC BY 4.0. https://github.com/babylonhealth/primock57

In [ ]:
!nvidia-smi

## 1. Settings & dependencies

In [ ]:
USE_DRIVE = True          # keep outputs on Google Drive
MODEL = "large-v3"        # or "large-v3-turbo" (faster), "medium", ...
RUN = "fw-" + MODEL       # name of the output / results folder
INITIAL_PROMPT = None     # e.g. "A primary care consultation between a GP and a patient."

import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/medasr"
else:
    WORK = "/content/medasr"
os.makedirs(f"{WORK}/scripts", exist_ok=True)
%cd {WORK}

In [ ]:
!pip install -q faster-whisper jiwer whisper_normalizer soundfile

Optional (untested): scispaCy medical entity extraction for the term-recall metric.
Without it, `evaluate.py` falls back to the bundled lexicon `scripts/medical_terms.txt`.
scispaCy pins an older spaCy, so this may downgrade Colab's spaCy.

In [ ]:
# !pip install -q scispacy https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_sm-0.5.4.tar.gz

## 2. Write the pipeline scripts (embedded copies)

In [ ]:
%%writefile scripts/download_data.py
"""Download PriMock57 (transcripts, notes, audio).

The GitHub repo stores the .wav files with Git LFS. A plain `git clone`
without git-lfs gives 133-byte pointer files instead of audio. This script:
  1. clones the repo with GIT_LFS_SKIP_SMUDGE=1 (fast; text files only),
  2. downloads each .wav from GitHub's LFS media endpoint
     (https://media.githubusercontent.com/media/babylonhealth/primock57/main/audio/<file>)
     and checks it against the sha256 + size in the pointer file.

Usage:
  python -I scripts/download_data.py --out data/primock57                 # all 57
  python -I scripts/download_data.py --out data/primock57 --limit 3       # first 3
  python -I scripts/download_data.py --out data/primock57 --ids day1_consultation01
"""
import argparse
import hashlib
import os
import subprocess
import sys
import urllib.request

REPO = "https://github.com/babylonhealth/primock57"
MEDIA = "https://media.githubusercontent.com/media/babylonhealth/primock57/main/audio/"


def read_pointer(path):
    """Return (sha256, size) if `path` is a Git LFS pointer file, else None."""
    if os.path.getsize(path) > 1024:
        return None
    with open(path, "r", errors="ignore") as f:
        text = f.read()
    if not text.startswith("version https://git-lfs.github.com/spec/v1"):
        return None
    fields = dict(line.split(" ", 1) for line in text.strip().splitlines())
    return fields["oid"].split(":", 1)[1], int(fields["size"])


def sha256_of(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--out", default="data/primock57")
    ap.add_argument("--ids", nargs="*", help="consultation ids, e.g. day1_consultation01")
    ap.add_argument("--limit", type=int, help="only the first N consultations (sorted)")
    args = ap.parse_args()

    if not os.path.isdir(os.path.join(args.out, "transcripts")):
        env = dict(os.environ, GIT_LFS_SKIP_SMUDGE="1")
        subprocess.run(["git", "clone", "--depth", "1", REPO, args.out], check=True, env=env)

    audio_dir = os.path.join(args.out, "audio")
    ids = sorted({f.rsplit("_", 1)[0] for f in os.listdir(audio_dir) if f.endswith(".wav")})
    if args.ids:
        ids = [i for i in ids if i in set(args.ids)]
    if args.limit:
        ids = ids[: args.limit]

    for cid in ids:
        for spk in ("doctor", "patient"):
            name = f"{cid}_{spk}.wav"
            path = os.path.join(audio_dir, name)
            ptr = read_pointer(path)
            if ptr is None:
                print(f"[ok]   {name} (already audio)")
                continue
            sha, size = ptr
            tmp = path + ".part"
            print(f"[get]  {name} ({size / 1e6:.1f} MB)")
            urllib.request.urlretrieve(MEDIA + name, tmp)
            if os.path.getsize(tmp) != size or sha256_of(tmp) != sha:
                os.remove(tmp)
                sys.exit(f"checksum mismatch for {name}")
            os.replace(tmp, path)
    print(f"done: {len(ids)} consultations in {audio_dir}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/prep_data.py
"""Prepare PriMock57 consultations for ASR evaluation.

For each consultation <cid> it writes to <out>/<cid>/:
  doctor.wav, patient.wav   16 kHz mono (copied, optionally cropped)
  mixed.wav                 doctor + patient summed, peak-normalised to -1 dBFS
                            (simulates one room microphone)
  ref_doctor.txt            reference text for the doctor channel
  ref_patient.txt           reference text for the patient channel
  ref_mixed.txt             both channels' utterances merged in start-time order
  ref_mixed_speakers.txt    same, one utterance per line with "Doctor:/Patient:" and times
  utterances.json           all utterances: speaker, start, end, raw text, clean text

Reference cleaning: <UNSURE>...</UNSURE> tags are removed but their words kept;
<UNIN/> and <INAUDIBLE_SPEECH/> markers are dropped. Further normalisation
(case, punctuation, numbers) happens in evaluate.py.

--max-seconds N crops each consultation for quick tests. The crop point is the
latest time <= N that falls in a gap of BOTH channels, so no utterance is cut
in half; only utterances ending before the crop point are kept.

Usage:
  python -I scripts/prep_data.py --src data/primock57 --out data/prepared
  python -I scripts/prep_data.py --src data/primock57 --out data/prepared_3min --max-seconds 180 --limit 3
"""
import argparse
import json
import os
import re

import numpy as np
import soundfile as sf

SR = 16000
DROP_TAGS = ["<UNSURE>", "</UNSURE>", "<UNIN/>", "<INAUDIBLE_SPEECH/>"]


def read_textgrid(path):
    """Minimal parser for Praat long-format TextGrid files (single IntervalTier).

    Returns a list of (xmin, xmax, text) for non-empty intervals."""
    with open(path, encoding="utf-8") as f:
        content = f.read()
    pattern = re.compile(
        r'intervals \[\d+\]:\s*xmin = ([\d.eE+-]+)\s*xmax = ([\d.eE+-]+)\s*text = "((?:[^"]|"")*)"',
        re.S,
    )
    out = []
    for xmin, xmax, text in pattern.findall(content):
        text = text.replace('""', '"').strip()
        if text:
            out.append((float(xmin), float(xmax), text))
    return out


def clean(text):
    for tag in DROP_TAGS:
        text = text.replace(tag, " ")
    return re.sub(r"\s+", " ", text).strip()


def find_crop_point(utts, max_s):
    """Latest utterance end <= max_s that is not inside any utterance (either speaker)."""
    for t in sorted({u["end"] for u in utts if u["end"] <= max_s}, reverse=True):
        if not any(u["start"] < t < u["end"] for u in utts):
            return t
    return max_s


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--src", default="data/primock57")
    ap.add_argument("--out", default="data/prepared")
    ap.add_argument("--ids", nargs="*")
    ap.add_argument("--limit", type=int)
    ap.add_argument("--max-seconds", type=float, help="crop each consultation (quick tests)")
    args = ap.parse_args()

    tdir = os.path.join(args.src, "transcripts")
    ids = sorted(f[: -len("_doctor.TextGrid")] for f in os.listdir(tdir) if f.endswith("_doctor.TextGrid"))
    if args.ids:
        ids = [i for i in ids if i in set(args.ids)]
    if args.limit:
        ids = ids[: args.limit]

    for cid in ids:
        wavs, utts = {}, []
        for spk in ("doctor", "patient"):
            wav_path = os.path.join(args.src, "audio", f"{cid}_{spk}.wav")
            if os.path.getsize(wav_path) < 1024:
                raise SystemExit(f"{wav_path} is a Git LFS pointer; run download_data.py first")
            audio, sr = sf.read(wav_path, dtype="float32")
            assert sr == SR and audio.ndim == 1, (wav_path, sr, audio.shape)
            wavs[spk] = audio
            for s, e, t in read_textgrid(os.path.join(tdir, f"{cid}_{spk}.TextGrid")):
                c = clean(t)
                if c:
                    utts.append({"speaker": spk, "start": s, "end": e, "raw": t, "text": c})
        utts.sort(key=lambda u: u["start"])

        # channels can differ by a few samples -> pad to the same length
        n = max(len(a) for a in wavs.values())
        for spk in wavs:
            wavs[spk] = np.pad(wavs[spk], (0, n - len(wavs[spk])))

        crop_s = None
        if args.max_seconds and n / SR > args.max_seconds:
            crop_s = find_crop_point(utts, args.max_seconds)
            utts = [u for u in utts if u["end"] <= crop_s]
            for spk in wavs:
                wavs[spk] = wavs[spk][: int(crop_s * SR)]

        mixed = wavs["doctor"] + wavs["patient"]
        peak = float(np.max(np.abs(mixed))) or 1.0
        mixed = mixed * (10 ** (-1 / 20) / peak)

        d = os.path.join(args.out, cid)
        os.makedirs(d, exist_ok=True)
        for name, audio in [("doctor", wavs["doctor"]), ("patient", wavs["patient"]), ("mixed", mixed)]:
            sf.write(os.path.join(d, f"{name}.wav"), audio, SR, subtype="PCM_16")
        for spk in ("doctor", "patient"):
            with open(os.path.join(d, f"ref_{spk}.txt"), "w") as f:
                f.write("\n".join(u["text"] for u in utts if u["speaker"] == spk) + "\n")
        with open(os.path.join(d, "ref_mixed.txt"), "w") as f:
            f.write("\n".join(u["text"] for u in utts) + "\n")
        with open(os.path.join(d, "ref_mixed_speakers.txt"), "w") as f:
            for u in utts:
                f.write(f"[{u['start']:7.2f}-{u['end']:7.2f}] {u['speaker'].capitalize()}: {u['text']}\n")
        meta = {"id": cid, "duration_s": len(mixed) / SR, "cropped_at_s": crop_s, "utterances": utts}
        with open(os.path.join(d, "utterances.json"), "w") as f:
            json.dump(meta, f, indent=1)
        n_words = sum(len(u["text"].split()) for u in utts)
        print(f"{cid}: {len(mixed) / SR:6.1f}s, {len(utts)} utterances, {n_words} ref words"
              + (f" (cropped at {crop_s:.1f}s)" if crop_s else ""))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/transcribe.py
"""Transcribe prepared PriMock57 audio with faster-whisper (default) or NVIDIA Parakeet.

Reads <data>/<cid>/{doctor,patient,mixed}.wav (from prep_data.py) and writes
<out>/<cid>/<condition>.json:
  {"id", "condition", "backend", "model", "initial_prompt", "audio_s",
   "runtime_s", "rtf", "text", "segments": [{"start", "end", "text", ...}]}
Existing outputs are skipped, so an interrupted run (e.g. a Colab disconnect)
can be resumed by re-running the same command.

Examples:
  # CPU smoke test
  python -I scripts/transcribe.py --data data/prepared_3min --out outputs/fw-base --model base
  # GPU (Colab)
  python -I scripts/transcribe.py --data data/prepared --out outputs/fw-large-v3 --model large-v3
  # with a domain prompt
  python -I scripts/transcribe.py ... --initial-prompt "A GP consultation between a doctor and a patient."
  # Parakeet (needs nemo_toolkit[asr]; untested in the smoke test)
  python -I scripts/transcribe.py --backend parakeet --data data/prepared --out outputs/parakeet
"""
import argparse
import json
import os
import time

import soundfile as sf


def load_faster_whisper(args):
    from faster_whisper import WhisperModel

    device = args.device
    if device == "auto":
        try:
            import ctranslate2
            device = "cuda" if ctranslate2.get_cuda_device_count() > 0 else "cpu"
        except Exception:
            device = "cpu"
    compute_type = args.compute_type or ("float16" if device == "cuda" else "int8")
    print(f"faster-whisper {args.model} on {device} ({compute_type})")
    model = WhisperModel(args.model, device=device, compute_type=compute_type,
                         cpu_threads=args.cpu_threads)

    def run(wav_path):
        segments, info = model.transcribe(
            wav_path,
            language="en",
            beam_size=args.beam_size,
            initial_prompt=args.initial_prompt,
            vad_filter=args.vad,
            condition_on_previous_text=not args.no_condition_on_previous_text,
        )
        out = []
        for s in segments:  # generator: decoding happens here
            out.append({
                "start": round(s.start, 2), "end": round(s.end, 2), "text": s.text.strip(),
                "avg_logprob": round(s.avg_logprob, 3),
                "no_speech_prob": round(s.no_speech_prob, 3),
                "compression_ratio": round(s.compression_ratio, 3),
            })
        return out

    return run


def load_parakeet(args):
    import nemo.collections.asr as nemo_asr

    name = args.model if args.model.startswith("nvidia/") else "nvidia/parakeet-tdt-0.6b-v2"
    print(f"NeMo {name}")
    model = nemo_asr.models.ASRModel.from_pretrained(model_name=name)
    if args.local_attention:
        # full attention handles ~24 min on an A100; local attention allows longer audio
        model.change_attention_model("rel_pos_local_attn", [256, 256])
        model.change_subsampling_conv_chunking_factor(1)

    def run(wav_path):
        hyp = model.transcribe([wav_path], timestamps=True)[0]
        segs = hyp.timestamp.get("segment", []) if getattr(hyp, "timestamp", None) else []
        if not segs:
            return [{"start": 0.0, "end": None, "text": hyp.text.strip()}]
        return [{"start": round(s["start"], 2), "end": round(s["end"], 2), "text": s["segment"].strip()}
                for s in segs]

    return run


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", default="data/prepared")
    ap.add_argument("--out", required=True)
    ap.add_argument("--backend", choices=["faster-whisper", "parakeet"], default="faster-whisper")
    ap.add_argument("--model", default="base",
                    help="faster-whisper: tiny/base/small/medium/large-v3/large-v3-turbo; "
                         "parakeet: nvidia/parakeet-tdt-0.6b-v2")
    ap.add_argument("--conditions", nargs="+", default=["doctor", "patient", "mixed"])
    ap.add_argument("--ids", nargs="*")
    ap.add_argument("--limit", type=int)
    ap.add_argument("--initial-prompt", default=None)
    ap.add_argument("--beam-size", type=int, default=5)
    ap.add_argument("--vad", action="store_true", help="faster-whisper Silero VAD filter (off by default)")
    ap.add_argument("--no-condition-on-previous-text", action="store_true")
    ap.add_argument("--device", default="auto")
    ap.add_argument("--compute-type", default=None)
    ap.add_argument("--cpu-threads", type=int, default=0)
    ap.add_argument("--local-attention", action="store_true", help="parakeet: for audio > ~20 min")
    args = ap.parse_args()

    ids = sorted(d for d in os.listdir(args.data) if os.path.isdir(os.path.join(args.data, d)))
    if args.ids:
        ids = [i for i in ids if i in set(args.ids)]
    if args.limit:
        ids = ids[: args.limit]

    run = load_parakeet(args) if args.backend == "parakeet" else load_faster_whisper(args)

    for cid in ids:
        os.makedirs(os.path.join(args.out, cid), exist_ok=True)
        for cond in args.conditions:
            out_path = os.path.join(args.out, cid, f"{cond}.json")
            if os.path.exists(out_path):
                print(f"skip {out_path}")
                continue
            wav = os.path.join(args.data, cid, f"{cond}.wav")
            audio_s = sf.info(wav).duration
            t0 = time.time()
            segments = run(wav)
            runtime = time.time() - t0
            result = {
                "id": cid, "condition": cond, "backend": args.backend, "model": args.model,
                "initial_prompt": args.initial_prompt, "audio_s": round(audio_s, 2),
                "runtime_s": round(runtime, 2), "rtf": round(runtime / audio_s, 4),
                "text": " ".join(s["text"] for s in segments),
                "segments": segments,
            }
            with open(out_path, "w") as f:
                json.dump(result, f, indent=1)
            print(f"{cid}/{cond}: {audio_s:.0f}s audio in {runtime:.1f}s "
                  f"({runtime / (audio_s / 60):.1f}s per audio-minute)")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/evaluate.py
"""Evaluate ASR outputs against PriMock57 references.

For every <hyp>/<cid>/<condition>.json (from transcribe.py) and the matching
<data>/<cid>/ref_<condition>.txt (from prep_data.py):

  * WER with jiwer, after Whisper's English text normaliser (lower-case,
    punctuation removed, fillers like "um"/"uh" removed, numbers and British
    spellings standardised). Texts are normalised utterance-by-utterance /
    segment-by-segment so hypothesis words keep their timestamps.
  * Medical-term recall = fraction of distinct medical terms in the reference
    that also occur (as a contiguous word sequence) in the hypothesis.
    Terms come from scispaCy en_core_sci_sm entities if that model is
    installed, otherwise from the hand-written lexicon scripts/medical_terms.txt
    (see --terms). The extractor used is recorded in the CSV.
  * Insertions: runs of hypothesis words that the jiwer alignment marks as
    "insert" (no reference word). Long runs are a cheap hallucination signal.

Writes to --results:
  wer.csv           one row per consultation x condition
  summary.csv       one row per condition (corpus WER = total errors / total ref words)
  insertions.csv    every insertion run with its timestamp and context
  missed_terms.csv  reference medical terms not found in the hypothesis

Usage:
  python -I scripts/evaluate.py --data data/prepared_3min --hyp outputs/fw-base --results results/fw-base
"""
import argparse
import csv
import glob
import json
import os
import re
from collections import defaultdict

import jiwer
from whisper_normalizer.english import EnglishTextNormalizer

NORM = EnglishTextNormalizer()
HERE = os.path.dirname(os.path.abspath(__file__))
# single-word scispaCy "entities" that are too generic to count as medical terms
SCI_STOP = {"yeah", "okay", "ok", "right", "thing", "things", "time", "times", "day", "days",
            "week", "weeks", "morning", "today", "bit", "lot", "sort", "kind", "way", "people",
            "sorry", "thank", "thanks", "good", "fine", "hello", "hi", "mean", "know", "like"}


def norm_words(text):
    return NORM(text).split()


def load_term_extractor(choice):
    if choice in ("auto", "scispacy"):
        try:
            import spacy
            nlp = spacy.load("en_core_sci_sm")

            def extract(raw_lines):
                terms = set()
                for line in raw_lines:
                    for ent in nlp(line).ents:
                        t = " ".join(norm_words(ent.text))
                        if t and t not in SCI_STOP:
                            terms.add(t)
                return terms

            return "scispacy:en_core_sci_sm", extract
        except Exception as e:  # not installed / model missing
            if choice == "scispacy":
                raise
            print(f"scispaCy unavailable ({type(e).__name__}: {e}); using lexicon fallback")

    lexicon = set()
    with open(os.path.join(HERE, "medical_terms.txt")) as f:
        for line in f:
            line = line.split("#", 1)[0].strip()
            if line:
                t = " ".join(norm_words(line))
                if t:
                    lexicon.add(t)

    def extract(raw_lines):
        words = norm_words(" ".join(raw_lines))
        return {t for t in lexicon if contains(words, t.split())}

    return f"lexicon:medical_terms.txt({len(lexicon)})", extract


def contains(words, term):
    """True if `term` (list of words) occurs contiguously in `words`; last word may carry a plural 's'."""
    n = len(term)
    for i in range(len(words) - n + 1):
        if words[i:i + n - 1] == term[:-1] and words[i + n - 1] in (term[-1], term[-1] + "s"):
            return True
    return False


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", default="data/prepared")
    ap.add_argument("--hyp", required=True, help="output folder of transcribe.py")
    ap.add_argument("--results", required=True)
    ap.add_argument("--terms", choices=["auto", "scispacy", "lexicon"], default="auto")
    ap.add_argument("--min-insert-words", type=int, default=1,
                    help="only log insertion runs with at least this many words")
    args = ap.parse_args()

    extractor_name, extract_terms = load_term_extractor(args.terms)
    os.makedirs(args.results, exist_ok=True)
    rows, ins_rows, miss_rows = [], [], []

    for hyp_path in sorted(glob.glob(os.path.join(args.hyp, "*", "*.json"))):
        with open(hyp_path) as f:
            hyp = json.load(f)
        cid, cond = hyp["id"], hyp["condition"]
        with open(os.path.join(args.data, cid, f"ref_{cond}.txt")) as f:
            ref_lines = [l.strip() for l in f if l.strip()]

        ref_words = [w for line in ref_lines for w in norm_words(line)]
        hyp_words, hyp_times = [], []
        for seg in hyp["segments"]:
            ws = norm_words(seg["text"])
            hyp_words += ws
            hyp_times += [seg["start"]] * len(ws)

        if not hyp_words:  # jiwer needs non-empty strings
            hyp_words, hyp_times = ["<empty>"], [0.0]
        out = jiwer.process_words(" ".join(ref_words), " ".join(hyp_words))

        for chunk in out.alignments[0]:
            if chunk.type != "insert":
                continue
            n = chunk.hyp_end_idx - chunk.hyp_start_idx
            if n < args.min_insert_words:
                continue
            a, b = chunk.hyp_start_idx, chunk.hyp_end_idx
            ins_rows.append({
                "id": cid, "condition": cond, "n_words": n,
                "segment_start_s": hyp_times[a],
                "inserted": " ".join(hyp_words[a:b]),
                "hyp_context": " ".join(hyp_words[max(0, a - 6):a]) + " [[" + " ".join(hyp_words[a:b])
                               + "]] " + " ".join(hyp_words[b:b + 6]),
                "ref_context": " ".join(ref_words[max(0, chunk.ref_start_idx - 6):chunk.ref_start_idx + 6]),
            })

        ref_terms = extract_terms(ref_lines)
        found = {t for t in ref_terms if contains(hyp_words, t.split())}
        for t in sorted(ref_terms - found):
            miss_rows.append({"id": cid, "condition": cond, "term": t})

        audio_min = hyp["audio_s"] / 60
        rows.append({
            "id": cid, "condition": cond, "backend": hyp["backend"], "model": hyp["model"],
            "initial_prompt": hyp.get("initial_prompt") or "",
            "ref_words": len(ref_words), "hyp_words": len(hyp_words),
            "wer": round(out.wer, 4), "substitutions": out.substitutions,
            "deletions": out.deletions, "insertions": out.insertions,
            "term_extractor": extractor_name, "ref_terms": len(ref_terms), "terms_found": len(found),
            "term_recall": round(len(found) / len(ref_terms), 4) if ref_terms else "",
            "audio_s": hyp["audio_s"], "runtime_s": hyp["runtime_s"],
            "sec_per_audio_min": round(hyp["runtime_s"] / audio_min, 2),
        })
        print(f"{cid}/{cond}: WER {out.wer:.3f}  term recall {len(found)}/{len(ref_terms)}")

    def write(name, data):
        if not data:
            return
        with open(os.path.join(args.results, name), "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=list(data[0].keys()))
            w.writeheader()
            w.writerows(data)

    by_cond = defaultdict(list)
    for r in rows:
        by_cond[r["condition"]].append(r)
    summary = []
    for cond, rs in by_cond.items():
        errs = sum(r["substitutions"] + r["deletions"] + r["insertions"] for r in rs)
        nref = sum(r["ref_words"] for r in rs)
        terms, found = sum(r["ref_terms"] for r in rs), sum(r["terms_found"] for r in rs)
        audio, runtime = sum(r["audio_s"] for r in rs), sum(r["runtime_s"] for r in rs)
        summary.append({
            "condition": cond, "model": rs[0]["model"], "n_consultations": len(rs),
            "audio_min": round(audio / 60, 2), "ref_words": nref,
            "corpus_wer": round(errs / nref, 4),
            "substitutions": sum(r["substitutions"] for r in rs),
            "deletions": sum(r["deletions"] for r in rs),
            "insertions": sum(r["insertions"] for r in rs),
            "term_recall_pooled": round(found / terms, 4) if terms else "",
            "sec_per_audio_min": round(runtime / (audio / 60), 2),
        })

    write("wer.csv", rows)
    write("summary.csv", summary)
    write("insertions.csv", ins_rows)
    write("missed_terms.csv", miss_rows)
    for s in summary:
        print(s)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/medical_terms.txt
# Fallback medical-term lexicon for evaluate.py (used when scispaCy/en_core_sci_sm
# is unavailable). Hand-written, general primary-care vocabulary; NOT derived
# from the PriMock57 transcripts. One term per line, '#' starts a comment.
# Terms are passed through the same Whisper English normaliser as the texts,
# so British/American spellings (diarrhoea/diarrhea) both work. A term also
# matches if its last word has a trailing "s" in the text (tablet -> tablets).
# Deliberately excludes very ambiguous everyday words (e.g. "back", "head",
# "cold", "chest" alone) to keep precision reasonable. Crude by design.

# --- symptoms / signs
pain
ache
headache
migraine
fever
temperature
chills
sweats
night sweats
cough
wheeze
wheezing
breathless
breathlessness
shortness of breath
chest pain
chest tightness
palpitations
dizziness
dizzy
lightheaded
fainting
faint
blackout
nausea
nauseous
vomiting
vomit
diarrhea
constipation
bloating
heartburn
indigestion
reflux
abdominal pain
stomach pain
tummy pain
cramps
cramping
blood in the stool
stool
stools
bowel movements
bowels
urine
urinating
urination
burning
itching
itchy
rash
swelling
swollen
lump
bruising
bleeding
discharge
fatigue
tired
tiredness
lethargic
lethargy
weakness
numbness
tingling
pins and needles
insomnia
weight loss
weight gain
appetite
sore throat
runny nose
blocked nose
congestion
earache
blurred vision
stiffness
tenderness
spasm
anxiety
anxious
depression
low mood
stress
panic attack
phlegm
sputum
mucus
thirst

# --- conditions / diagnoses
infection
viral infection
bacterial infection
urinary tract infection
uti
chest infection
gastroenteritis
food poisoning
flu
influenza
covid
asthma
copd
eczema
psoriasis
dermatitis
acne
allergy
allergies
allergic
hay fever
diabetes
type 2 diabetes
hypertension
high blood pressure
low blood pressure
high cholesterol
cholesterol
heart attack
angina
stroke
arrhythmia
atrial fibrillation
heart failure
thyroid
hypothyroidism
hyperthyroidism
arthritis
osteoarthritis
gout
sciatica
back pain
lower back pain
sprain
fracture
tendonitis
irritable bowel syndrome
ibs
crohn
colitis
appendicitis
gallstones
ulcer
hernia
kidney stones
cystitis
pneumonia
bronchitis
sinusitis
tonsillitis
conjunctivitis
glandular fever
anemia
cancer
tumor
epilepsy
seizure
concussion
dementia
pregnancy
pregnant
period
periods
menstrual
menopause
contraception
sexually transmitted infection
std

# --- anatomy
abdomen
stomach
tummy
bowel
bladder
kidney
kidneys
liver
lungs
heart
throat
tonsils
sinuses
ear
ears
eye
eyes
skin
joint
joints
knee
ankle
wrist
elbow
shoulder
hip
spine
neck
muscle
muscles
lower abdomen
left side
right side
groin
pelvis
rectum
chest wall
calf
thigh

# --- medications / treatments
medication
medications
medicine
tablet
pill
capsule
inhaler
blue inhaler
brown inhaler
steroid
steroids
antibiotic
antibiotics
amoxicillin
penicillin
paracetamol
acetaminophen
ibuprofen
aspirin
naproxen
codeine
co codamol
tramadol
morphine
omeprazole
lansoprazole
gaviscon
antacid
antihistamine
cetirizine
loratadine
salbutamol
ventolin
prednisolone
hydrocortisone
metformin
insulin
statin
statins
atorvastatin
simvastatin
amlodipine
ramipril
lisinopril
levothyroxine
sertraline
citalopram
fluoxetine
antidepressant
antidepressants
contraceptive pill
the pill
rehydration
dioralyte
oral rehydration
fluids
cream
ointment
moisturizer
emollient
eye drops
nasal spray
vaccine
vaccination
dose
dosage
milligrams
mg
side effects
over the counter
prescription
physiotherapy
surgery
operation

# --- tests / procedures / measurements
blood test
blood tests
urine test
urine sample
stool sample
swab
x ray
scan
ultrasound
ct scan
mri
ecg
blood pressure
heart rate
pulse
oxygen
temperature
examination
referral
specialist
hospital
a and e
emergency
ambulance
gp
pharmacist
pharmacy
nurse
follow up
smear test

# --- history / lifestyle
smoke
smoking
smoker
cigarettes
alcohol
drink
units
exercise
diet
family history
medical history
past medical history
allergic to

## 3. Download PriMock57 (~2 GB audio) and prepare the three conditions

In [ ]:
!python -I scripts/download_data.py --out data/primock57

In [ ]:
!python -I scripts/prep_data.py --src data/primock57 --out data/prepared

## 4. Transcribe (all 57 consultations x doctor / patient / mixed)
About 26 h of audio in total (8.6 h per condition). Expect this to take hours on a T4; it resumes if interrupted.

In [ ]:
prompt_arg = f'--initial-prompt "{INITIAL_PROMPT}"' if INITIAL_PROMPT else ""
!python -I scripts/transcribe.py --data data/prepared --out outputs/{RUN} --model {MODEL} {prompt_arg}

## 5. Evaluate

In [ ]:
!python -I scripts/evaluate.py --data data/prepared --hyp outputs/{RUN} --results results/{RUN}

In [ ]:
import pandas as pd
summary = pd.read_csv(f"results/{RUN}/summary.csv"); display(summary)
wer = pd.read_csv(f"results/{RUN}/wer.csv")
display(wer.pivot(index="id", columns="condition", values="wer").describe())
ins = pd.read_csv(f"results/{RUN}/insertions.csv")
display(ins.sort_values("n_words", ascending=False).head(20))   # longest insertion runs = hallucination candidates

In [ ]:
!cd results && zip -qr /content/{RUN}_results.zip {RUN} && echo saved /content/{RUN}_results.zip

## Optional: NVIDIA Parakeet TDT 0.6B v2 (untested)
Installing NeMo takes several minutes. Consultations are up to ~14 min; if a T4 runs out of
memory, add `--local-attention`.

In [ ]:
# !pip install -q "nemo_toolkit[asr]"
# !python -I scripts/transcribe.py --backend parakeet --model nvidia/parakeet-tdt-0.6b-v2 --data data/prepared --out outputs/parakeet
# !python -I scripts/evaluate.py --data data/prepared --hyp outputs/parakeet --results results/parakeet